In [145]:
import os
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import re
import openpyxl
import numpy as np

In [146]:
JNPFile=r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\Import files\JNP\CompleteJNP_Data.csv"

In [147]:
df_JNP=pd.read_csv(JNPFile)

In [148]:
df_JNP['Key']=df_JNP['Article Number']+df_JNP['Product Group']+df_JNP['Attribute']

In [149]:
df_PNS=df_JNP[['Article Number','Product Group']].drop_duplicates(ignore_index=True)

In [150]:
df_PNS

,Article Number,Product Group
0,A99000,Wheel Hub Washer
1,D1034,Parking Brake Disc Brake Pad Set
2,19623,Parking Brake Cable
3,91825,Parking Brake Cable
4,91901,Parking Brake Cable
...,...,...
38420,9528B,Wheel Lug Stud
38421,9530B,Wheel Lug Stud
38422,9550B,Wheel Lug Stud
38423,9585B,Wheel Lug Stud


## AutocareList

In [151]:
df_AC=pd.read_csv(r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\Autocare\Reference\df_full.csv")

In [152]:
df_AC=df_AC[[ 'PartTerminologyName','PAName', 'UOMLabel']]

In [153]:
df_AC['PAName'][1]

'Width'

In [154]:
df_AC.dropna(subset=['PAName']).reset_index(drop=True)

,PartTerminologyName,PAName,UOMLabel
0,Car Cover,Length,m
1,Car Cover,Width,m
2,Car Cover,Length,in
3,Car Cover,Width,in
4,Car Cover,Length,ft
...,...,...,...
830894,NaN,Oil Supply Line Included,NaN
830895,NaN,Oil Drain Line Included,NaN
830896,NaN,Coolant Supply Line Included,NaN
830897,NaN,Coolant Drain Line Included,NaN


In [155]:
df_AC["Attribute_Fullname"] = np.where(df_AC["UOMLabel"].notna(), df_AC['PAName'] + " ("+df_AC['UOMLabel']+")", df_AC["PAName"])

In [156]:
df_AC=df_AC.drop_duplicates(ignore_index=True)

In [157]:
df_AC['Autocare']="Autocare_Attribute"

In [158]:
df_AC

,PartTerminologyName,PAName,UOMLabel,Attribute_Fullname,Autocare
0,Car Cover,Length,m,Length (m),Autocare_Attribute
1,Car Cover,Width,m,Width (m),Autocare_Attribute
2,Car Cover,Length,in,Length (in),Autocare_Attribute
3,Car Cover,Width,in,Width (in),Autocare_Attribute
4,Car Cover,Length,ft,Length (ft),Autocare_Attribute
...,...,...,...,...,...
210461,NaN,Oil Supply Line Included,NaN,Oil Supply Line Included,Autocare_Attribute
210462,NaN,Oil Drain Line Included,NaN,Oil Drain Line Included,Autocare_Attribute
210463,NaN,Coolant Supply Line Included,NaN,Coolant Supply Line Included,Autocare_Attribute
210464,NaN,Coolant Drain Line Included,NaN,Coolant Drain Line Included,Autocare_Attribute


## Merging

In [159]:
df_Listed=df_PNS.merge(df_AC,how='left',left_on='Product Group',right_on='PartTerminologyName')


In [160]:
df_Listed=df_Listed[['Article Number','Product Group', 'Attribute_Fullname',"Autocare"]]
df_Listed

,Article Number,Product Group,Attribute_Fullname,Autocare
0,A99000,Wheel Hub Washer,Thickness (mm),Autocare_Attribute
1,A99000,Wheel Hub Washer,Outside Diameter (mm),Autocare_Attribute
2,A99000,Wheel Hub Washer,Inside Diameter (mm),Autocare_Attribute
3,A99000,Wheel Hub Washer,Thickness (in),Autocare_Attribute
4,A99000,Wheel Hub Washer,Outside Diameter (in),Autocare_Attribute
...,...,...,...,...
1643337,9822B,Wheel Lug Stud,Shoulder Shape,Autocare_Attribute
1643338,9822B,Wheel Lug Stud,Grade Type,Autocare_Attribute
1643339,9822B,Wheel Lug Stud,Thread Direction,Autocare_Attribute
1643340,9822B,Wheel Lug Stud,Bolt Grade,Autocare_Attribute


In [161]:
df_Listed['Key']=df_Listed['Article Number']+df_Listed['Product Group']+df_Listed['Attribute_Fullname']

In [162]:
df_Final=df_Listed.merge(df_JNP,how="outer")

In [163]:
df_Final['Attributes']=np.where(df_Final["Attribute_Fullname"].notna(), df_Final["Attribute_Fullname"] , df_Final["Attribute"])

In [164]:
df_Final=df_Final[['Article Number', 'Product Group', 'Attributes', 'Autocare', 'Value']]

In [165]:
df_Final['AC_AttributeCount']=np.where(df_Final["Autocare"].notna(), 1 , 0)

In [166]:
df_Final=df_Final.dropna(subset='Autocare')

In [167]:
df_Final['Attribute_Value']=np.where((df_Final["Autocare"].notna())  & (df_Final["Value"].notna()), 1 , 0)

In [168]:
df_Final

,Article Number,Product Group,Attributes,Autocare,Value,AC_AttributeCount,Attribute_Value
0,0158B,Wheel Lug Stud,Bolt Grade,Autocare_Attribute,NaN,1,0
2,0158B,Wheel Lug Stud,Color,Autocare_Attribute,NaN,1,0
3,0158B,Wheel Lug Stud,Grade Type,Autocare_Attribute,NaN,1,0
6,0158B,Wheel Lug Stud,Length (in),Autocare_Attribute,NaN,1,0
7,0158B,Wheel Lug Stud,Length (mm),Autocare_Attribute,NaN,1,0
...,...,...,...,...,...,...,...
3578470,X420R,Disc Brake Pad Set,Prepared For Pad Wear Sensor,Autocare_Attribute,NaN,1,0
3578472,X420R,Disc Brake Pad Set,Slotted,Autocare_Attribute,NaN,1,0
3578473,X420R,Disc Brake Pad Set,Wear Indicator Attached,Autocare_Attribute,NaN,1,0
3578474,X420R,Disc Brake Pad Set,Weight (kg),Autocare_Attribute,NaN,1,0


In [169]:
df_1 = df_Final.iloc[:1000000,:]
df_2 = df_Final.iloc[1000000:,:]

In [170]:
df_Final=df_Final.groupby(['Article Number','Product Group'], as_index=False).agg(
    Total_Attr=('AC_AttributeCount','sum'),
    Filled_Attr=('Attribute_Value','sum'))

In [171]:
df_Final['Filled%']=df_Final['Filled_Attr']/df_Final['Total_Attr']

In [172]:
df_Final

,Article Number,Product Group,Total_Attr,Filled_Attr,Filled%
0,0158B,Wheel Lug Stud,17,0,0.000000
1,0159B,Wheel Lug Stud,17,0,0.000000
2,0505B,Wheel Lug Stud,17,0,0.000000
3,0522B,Wheel Lug Stud,17,0,0.000000
4,0523B,Wheel Lug Stud,17,1,0.058824
...,...,...,...,...,...
38420,WVA24608,Disc Brake Pad Set,42,0,0.000000
38421,X420F,Disc Brake Pad Set,42,0,0.000000
38422,X420FA,Disc Brake Pad Set,42,0,0.000000
38423,X420FB,Disc Brake Pad Set,42,0,0.000000


## Exporting

In [174]:
with pd.ExcelWriter(r'C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\Standard\Import files\JNP\BPI_SKU_AttrPerc.xlsx') as writer:  # doctest: +SKIP
    df_1.to_excel(writer,index=False, sheet_name='Raw1')
    df_2.to_excel(writer,index=False, sheet_name='Raw2')
    df_Final.to_excel(writer,index=False, sheet_name='Fillinginfo')